In [ ]:
import os
os.environ["PYTENSOR_FLAGS"] = "cxx=" 

import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import matplotlib as plt

# Load and slice 
df = pd.read_csv('../data/BrentOilPrices.csv')
df['Date'] = pd.to_datetime(df['Date'], format='mixed')
df_fast = df[df['Date'] >= '2020-01-01'].iloc[::3].copy().reset_index(drop=True)

prices = df_fast['Price'].values
n_data = len(prices)
t = np.arange(n_data)

with pm.Model() as model:
    # Priors
    tau = pm.Normal("tau", mu=n_data / 2, sigma=n_data / 10)
    mu_1 = pm.Normal("mu_1", mu=prices.mean(), sigma=15)
    mu_2 = pm.Normal("mu_2", mu=prices.mean(), sigma=15)
    sigma = pm.HalfNormal("sigma", sigma=10)
    
    # Simple switch
    mu_regime = pm.math.switch(tau > t, mu_1, mu_2)
    obs = pm.Normal("obs", mu=mu_regime, sigma=sigma, observed=prices)
    
    # SLICE SAMPLER: This is much faster when you don't have a C++ compiler
    # because it doesn't need to do complex gradient calculus.
    step = pm.Slice()
    trace = pm.sample(500, tune=500, step=step, chains=1, cores=1)

# Results
tau_samples = trace.posterior["tau"].values.flatten()
most_likely_idx = int(np.median(tau_samples))
change_date = df_fast.iloc[max(0, min(n_data-1, most_likely_idx))]['Date']

print(f"\nSuccess! Change detected around: {change_date.date()}")

Sequential sampling (1 chains in 1 job)
CompoundStep
>Slice: [tau]
>Slice: [mu_1]
>Slice: [mu_2]
>Slice: [sigma]


c:\Users\a_kas\brent-oil-change-point-analysis\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [ ]:
import arviz as az
# Check convergence (Fuzzy caterpillars)
az.plot_trace(trace, var_names=["tau", "mu_1", "mu_2"])
plt.show()

# Get summary stats
summary = az.summary(trace, var_names=["mu_1", "mu_2"])
print(summary)

# Identify the Date
tau_samples = trace.posterior["tau"].values.flatten()
most_likely_idx = int(np.median(tau_samples))
change_date = df_recent.iloc[most_likely_idx]['Date']

print(f"\n--- RESULTS ---")
print(f"Detected Change Point: {change_date.date()}")
print(f"Price shifted from ${summary.loc['mu_1', 'mean']:.2f} to ${summary.loc['mu_2', 'mean']:.2f}")

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(df_recent['Date'], prices, label="Actual Price", alpha=0.5)

# Plot the identified means
plt.axvline(change_date, color='red', linestyle='--', label=f"Switch Point: {change_date.date()}")
plt.hlines(summary.loc['mu_1', 'mean'], xmin=df_recent['Date'].min(), xmax=change_date, color='green', label="Pre-switch Mean")
plt.hlines(summary.loc['mu_2', 'mean'], xmin=change_date, xmax=df_recent['Date'].max(), color='orange', label="Post-switch Mean")

plt.title("Bayesian Change Point Analysis: Brent Oil (2014-2022)")
plt.legend()
plt.savefig('../reports/figures/04_change_point_result.png')
plt.show()